# 08. 주모형 결과

이 노트북은 10주차 Pivot 이후 새 가설 구조에 맞춰 주모형 결과를 산출한다. 분석의 핵심 구조는 프로세스 개선 관련 정보화 효과 인식(`effect_proc_improve`)을 종속변수로 두고, AI 활용 범위(`ai_use_sum`)와 조직적/디지털 보완재의 조건부 관련성을 확인하는 것이다.

이번 노트북에서는 **주모형 OLS 결과만** 다룬다. 모든 OLS 모형에는 HC3 robust standard error를 적용한다. Ordered Logit, 대체 DV, weighted/unweighted 비교, DMI 하위집단 분석, 기타 강건성 분석은 다음 노트북에서 별도로 수행한다.

특히 교수님 피드백을 반영하여, **9주차 중간과제의 Model D 결과를 Pivot 이후 H2의 headline model로 재배치**한다. 따라서 Model 3은 정보화 투자 범위(`it_invest_sum`)를 통제한 뒤에도 `ai_use_sum:it_org_any` 상호작용항이 유지되는지를 확인하는 H2 핵심 모형이다.

## 1. 설정 및 라이브러리

In [19]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

try:
    from IPython.display import display, Markdown
except ImportError:
    class Markdown(str):
        pass

    def display(obj):
        print(obj)

SAVE_OUTPUTS = False

BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name == "code":
    BASE_DIR = BASE_DIR.parent

DATA_CANDIDATES = [
    BASE_DIR / "working" / "analysis" / "nia_2024_analysis_total.csv",
    BASE_DIR / "working" / "featured" / "nia_2024_featured.csv",
    BASE_DIR / "working" / "cleaned" / "nia_2024_cleaned.csv",
]

OUTPUT_DIR = BASE_DIR / Path("outputs/08_main_model")
TABLE_DIR = OUTPUT_DIR / "tables"
MODEL_DIR = OUTPUT_DIR / "models"

if SAVE_OUTPUTS:
    TABLE_DIR.mkdir(parents=True, exist_ok=True)
    MODEL_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 200)

print("BASE_DIR:", BASE_DIR)
print("SAVE_OUTPUTS:", SAVE_OUTPUTS)
print("OUTPUT_DIR:", OUTPUT_DIR)

BASE_DIR: /Users/yenarue/Downloads/ITM.89912(A)/연구 데이터
SAVE_OUTPUTS: False
OUTPUT_DIR: /Users/yenarue/Downloads/ITM.89912(A)/연구 데이터/outputs/08_main_model


## 2. 데이터 로드 및 변수 확인

In [20]:
existing_candidates = [path for path in DATA_CANDIDATES if path.exists()]
if not existing_candidates:
    candidate_text = "\n".join(f"- {path}" for path in DATA_CANDIDATES)
    raise FileNotFoundError("분석 데이터 파일을 찾지 못했습니다. 아래 후보 경로를 확인하세요:\n" + candidate_text)

DATA_PATH = existing_candidates[0]
df = pd.read_csv(DATA_PATH, low_memory=False)
LOADED_SHAPE = df.shape

print("사용 데이터 파일:", DATA_PATH.relative_to(BASE_DIR))
print("데이터 shape:", LOADED_SHAPE)

사용 데이터 파일: working/analysis/nia_2024_analysis_total.csv
데이터 shape: (12203, 103)


## 3. 변수 처리

In [21]:
DV = "effect_proc_improve"
AI = "ai_use_sum"
IT_ORG = "it_org_any"
DMI = "dmi"
INVEST_CONTROL = "it_invest_sum"
BASE_CONTROLS = ["firm_size", "industry", "region", "firm_type"]

CHECK_VARS = [DV, AI, IT_ORG, DMI, INVEST_CONTROL] + BASE_CONTROLS

variable_check = pd.DataFrame([
    {
        "변수명": var,
        "존재 여부": "있음" if var in df.columns else "없음",
        "유효 N": int(df[var].notna().sum()) if var in df.columns else np.nan,
        "결측 N": int(df[var].isna().sum()) if var in df.columns else np.nan,
        "dtype": str(df[var].dtype) if var in df.columns else "해당 없음",
    }
    for var in CHECK_VARS
])

display(variable_check)

missing_required = variable_check.loc[variable_check["존재 여부"] != "있음", "변수명"].tolist()
missing_base_controls = [var for var in BASE_CONTROLS if var not in df.columns]
missing_core_controls = [var for var in ["firm_size", "industry"] if var not in df.columns]

if missing_base_controls:
    warnings.warn("일부 통제변수가 없어 해당 변수는 제외합니다: " + ", ".join(missing_base_controls))
if missing_core_controls:
    warnings.warn("핵심 통제변수 누락: " + ", ".join(missing_core_controls))

required_for_main = [DV, AI, IT_ORG, DMI, INVEST_CONTROL, "firm_size", "industry"]
missing_for_main = [var for var in required_for_main if var not in df.columns]
if missing_for_main:
    display(Markdown("**필수 변수가 없어 주모형 추정을 중단합니다:** " + ", ".join(missing_for_main)))
    raise SystemExit("Missing required variables: " + ", ".join(missing_for_main))

available_controls = [var for var in BASE_CONTROLS if var in df.columns]
print("사용 통제변수:", available_controls)
display(Markdown("**필수 변수 확인 결과:** 주모형 추정에 필요한 핵심 변수가 존재합니다."))

,변수명,존재 여부,유효 N,결측 N,dtype
0,effect_proc_improve,있음,12203,0,int64
1,ai_use_sum,있음,12203,0,int64
2,it_org_any,있음,12203,0,int64
3,dmi,있음,12203,0,int64
4,it_invest_sum,있음,12203,0,int64
5,firm_size,있음,12203,0,int64
6,industry,있음,12203,0,int64
7,region,있음,12203,0,int64
8,firm_type,있음,12203,0,int64


사용 통제변수: ['firm_size', 'industry', 'region', 'firm_type']


**필수 변수 확인 결과:** 주모형 추정에 필요한 핵심 변수가 존재합니다.

### 3.1 분석 데이터 구성

결측이 있는 행은 해당 모형에 필요한 변수 기준으로 listwise deletion된다. `firm_size`는 이번 주모형에서 순서형 통제변수로 그대로 포함하고, `industry`, `region`, `firm_type`은 formula에서 `C()`로 처리한다.

In [22]:
needed_vars = sorted(set([DV, AI, IT_ORG, DMI, INVEST_CONTROL] + available_controls))
analysis_df = df[needed_vars].copy()

for var in needed_vars:
    analysis_df[var] = pd.to_numeric(analysis_df[var], errors="coerce")

missing_summary = pd.DataFrame({
    "변수명": needed_vars,
    "유효 N": [int(analysis_df[var].notna().sum()) for var in needed_vars],
    "결측 N": [int(analysis_df[var].isna().sum()) for var in needed_vars],
})
display(missing_summary)
print("전체 주모형 변수 complete-case N:", f"{analysis_df.dropna().shape[0]:,}")

,변수명,유효 N,결측 N
0,ai_use_sum,12203,0
1,dmi,12203,0
2,effect_proc_improve,12203,0
3,firm_size,12203,0
4,firm_type,12203,0
5,industry,12203,0
6,it_invest_sum,12203,0
7,it_org_any,12203,0
8,region,12203,0


전체 주모형 변수 complete-case N: 12,203


## 4. 회귀식 구성

아래 5개 모형을 OLS HC3로 추정한다. Model 3은 **H2 Headline: 9주차 Model D 재현 결과**로, 정보화 투자 범위(`it_invest_sum`)를 통제한 뒤에도 `ai_use_sum:it_org_any` 상호작용항이 유지되는지를 확인한다.

In [23]:
def control_formula(controls: list[str]) -> str:
    terms = []
    for var in controls:
        if var == "firm_size":
            terms.append("firm_size")
        elif var in ["industry", "region", "firm_type"]:
            terms.append(f"C({var})")
        else:
            terms.append(var)
    return " + ".join(terms)

CONTROL_TERMS = control_formula(available_controls)
CONTROL_SUFFIX = f" + {CONTROL_TERMS}" if CONTROL_TERMS else ""

MODEL_SPECS = {
    "M1_H1_Baseline": {
        "short": "Model 1",
        "label": "H1 Baseline",
        "column": "Model 1: H1 Baseline",
        "formula": f"{DV} ~ {AI}{CONTROL_SUFFIX}",
        "hypothesis": "H1",
        "investment_control": "No",
    },
    "M2_H2_Basic_AI_ITorg": {
        "short": "Model 2",
        "label": "H2 Basic: AI × IT org",
        "column": "Model 2: H2 Basic: AI × IT org",
        "formula": f"{DV} ~ {AI} + {IT_ORG} + {AI}:{IT_ORG}{CONTROL_SUFFIX}",
        "hypothesis": "H2 Basic",
        "investment_control": "No",
    },
    "M3_H2_Headline_Model_D": {
        "short": "Model 3",
        "label": "H2 Headline: Model D replication",
        "column": "Model 3: H2 Headline: Model D replication",
        "formula": f"{DV} ~ {AI} + {IT_ORG} + {INVEST_CONTROL} + {AI}:{IT_ORG}{CONTROL_SUFFIX}",
        "hypothesis": "H2 Headline",
        "investment_control": "Yes",
    },
    "M4_H3_AI_DMI": {
        "short": "Model 4",
        "label": "H3: AI × DMI",
        "column": "Model 4: H3: AI × DMI",
        "formula": f"{DV} ~ {AI} + {DMI} + {AI}:{DMI}{CONTROL_SUFFIX}",
        "hypothesis": "H3",
        "investment_control": "No",
    },
    "M5_Integrated_Complements": {
        "short": "Model 5",
        "label": "Integrated Complementarity Model",
        "column": "Model 5: Integrated complements",
        "formula": f"{DV} ~ {AI} + {IT_ORG} + {DMI} + {INVEST_CONTROL} + {AI}:{IT_ORG} + {AI}:{DMI}{CONTROL_SUFFIX}",
        "hypothesis": "Integrated",
        "investment_control": "Yes",
    },
}

model_spec_table = pd.DataFrame([
    {
        "모형": spec["short"],
        "가설/역할": spec["label"],
        "Investment control 포함": spec["investment_control"],
        "formula": spec["formula"],
    }
    for spec in MODEL_SPECS.values()
])
display(model_spec_table)

,모형,가설/역할,Investment control 포함,formula
0,Model 1,H1 Baseline,No,effect_proc_improve ~ ai_use_sum + firm_size +...
1,Model 2,H2 Basic: AI × IT org,No,effect_proc_improve ~ ai_use_sum + it_org_any ...
2,Model 3,H2 Headline: Model D replication,Yes,effect_proc_improve ~ ai_use_sum + it_org_any ...
3,Model 4,H3: AI × DMI,No,effect_proc_improve ~ ai_use_sum + dmi + ai_us...
4,Model 5,Integrated Complementarity Model,Yes,effect_proc_improve ~ ai_use_sum + it_org_any ...


## 5. 공통 함수

In [24]:
def p_stars(p_value: float) -> str:
    if pd.isna(p_value):
        return ""
    if p_value < 0.001:
        return "***"
    if p_value < 0.01:
        return "**"
    if p_value < 0.05:
        return "*"
    return ""


def direction(coef: float) -> str:
    if pd.isna(coef):
        return ""
    return "+" if coef > 0 else "-" if coef < 0 else "0"


def format_coef_with_stars(coef: float, se: float, p_value: float) -> str:
    if pd.isna(coef):
        return ""
    return f"{coef:.3f}{p_stars(p_value)}\n({se:.3f})"


def variables_in_formula(spec: dict) -> list[str]:
    base = [DV, AI]
    formula = spec["formula"]
    for var in [IT_ORG, DMI, INVEST_CONTROL] + available_controls:
        if var in formula and var not in base:
            base.append(var)
    return base


def fit_ols_hc3(model_key: str, spec: dict):
    model_vars = variables_in_formula(spec)
    model_data = analysis_df[model_vars].dropna().copy()
    if model_data.empty:
        warnings.warn(f"{model_key} 추정에 사용할 complete-case 데이터가 없습니다.")
        return None, model_data
    result = smf.ols(spec["formula"], data=model_data).fit(cov_type="HC3")
    return result, model_data


def term_result(model, term: str) -> dict:
    if model is None or term not in model.params.index:
        return {
            "계수": np.nan, "HC3 Robust SE": np.nan, "p-value": np.nan, "유의수준": "",
            "방향": "", "formatted": "",
        }
    coef = float(model.params[term])
    se = float(model.bse[term])
    p_value = float(model.pvalues[term])
    return {
        "계수": coef,
        "HC3 Robust SE": se,
        "p-value": p_value,
        "유의수준": p_stars(p_value),
        "방향": direction(coef),
        "formatted": format_coef_with_stars(coef, se, p_value),
    }


def extract_key_results(model_key: str, model, spec: dict, term: str, hypothesis: str, note: str) -> dict:
    tr = term_result(model, term)
    return {
        "가설": hypothesis,
        "모형": spec["label"],
        "종속변수": DV,
        "핵심 변수": term,
        "계수": tr["계수"],
        "HC3 Robust SE": tr["HC3 Robust SE"],
        "p-value": tr["p-value"],
        "유의수준": tr["유의수준"],
        "N": int(model.nobs) if model is not None else np.nan,
        "R²": model.rsquared if model is not None else np.nan,
        "Adj. R²": model.rsquared_adj if model is not None else np.nan,
        "해석 방향": tr["방향"],
        "비고": note,
    }


def display_note(title: str, lines: list[str]) -> None:
    display(Markdown("**" + title + "**\n" + "\n".join(f"- {line}" for line in lines)))

## 6. 모델 추정

In [25]:
MODELS = {}
MODEL_DATA = {}
fit_rows = []

for model_key, spec in MODEL_SPECS.items():
    result, model_data = fit_ols_hc3(model_key, spec)
    MODELS[model_key] = result
    MODEL_DATA[model_key] = model_data
    if result is None:
        print(f"{spec['short']} {spec['label']}: 추정 실패")
        continue
    print(f"{spec['short']} {spec['label']} 추정 완료: N={int(result.nobs):,}, R²={result.rsquared:.3f}, Adj. R²={result.rsquared_adj:.3f}")
    fit_rows.append({
        "모형": spec["short"],
        "가설/역할": spec["label"],
        "N": int(result.nobs),
        "R²": result.rsquared,
        "Adj. R²": result.rsquared_adj,
        "AIC": result.aic,
        "BIC": result.bic,
        "HC3 robust SE 적용 여부": "Yes",
        "Controls 포함 여부": "Yes" if available_controls else "No",
        "Industry FE 포함 여부": "Yes" if "industry" in available_controls else "No",
        "Region FE 포함 여부": "Yes" if "region" in available_controls else "No",
        "Firm type FE 포함 여부": "Yes" if "firm_type" in available_controls else "No",
        "Investment control 포함 여부": spec["investment_control"],
    })

main_model_fit_stats = pd.DataFrame(fit_rows)
for col in ["R²", "Adj. R²", "AIC", "BIC"]:
    main_model_fit_stats[col] = main_model_fit_stats[col].round(3)

display(main_model_fit_stats)

Model 1 H1 Baseline 추정 완료: N=12,203, R²=0.112, Adj. R²=0.110
Model 2 H2 Basic: AI × IT org 추정 완료: N=12,203, R²=0.125, Adj. R²=0.122
Model 3 H2 Headline: Model D replication 추정 완료: N=12,203, R²=0.148, Adj. R²=0.145
Model 4 H3: AI × DMI 추정 완료: N=12,203, R²=0.126, Adj. R²=0.124
Model 5 Integrated Complementarity Model 추정 완료: N=12,203, R²=0.151, Adj. R²=0.148


,모형,가설/역할,N,R²,Adj. R²,AIC,BIC,HC3 robust SE 적용 여부,Controls 포함 여부,Industry FE 포함 여부,Region FE 포함 여부,Firm type FE 포함 여부,Investment control 포함 여부
0,Model 1,H1 Baseline,12203,0.112,0.110,26682.802,26942.133,Yes,Yes,Yes,Yes,Yes,No
1,Model 2,H2 Basic: AI × IT org,12203,0.125,0.122,26517.309,26791.458,Yes,Yes,Yes,Yes,Yes,No
2,Model 3,H2 Headline: Model D replication,12203,0.148,0.145,26190.144,26471.702,Yes,Yes,Yes,Yes,Yes,Yes
3,Model 4,H3: AI × DMI,12203,0.126,0.124,26493.169,26767.319,Yes,Yes,Yes,Yes,Yes,No
4,Model 5,Integrated Complementarity Model,12203,0.151,0.148,26153.767,26450.145,Yes,Yes,Yes,Yes,Yes,Yes


## 7. Full summary 확인

In [26]:
for model_key in ["M1_H1_Baseline","M2_H2_Basic_AI_ITorg","M4_H3_AI_DMI","M3_H2_Headline_Model_D", "M5_Integrated_Complements"]:
    model = MODELS.get(model_key)
    spec = MODEL_SPECS[model_key]
    if model is not None:
        print("\n" + "=" * 100)
        print(f"{spec['short']} {spec['label']} full summary")
        print(spec["formula"])
        print(model.summary())


Model 1 H1 Baseline full summary
effect_proc_improve ~ ai_use_sum + firm_size + C(industry) + C(region) + C(firm_type)
                             OLS Regression Results                            
Dep. Variable:     effect_proc_improve   R-squared:                       0.112
Model:                             OLS   Adj. R-squared:                  0.110
Method:                  Least Squares   F-statistic:                     48.95
Date:                 Sat, 16 May 2026   Prob (F-statistic):          6.41e-307
Time:                         20:13:19   Log-Likelihood:                -13306.
No. Observations:                12203   AIC:                         2.668e+04
Df Residuals:                    12168   BIC:                         2.694e+04
Df Model:                           34                                         
Covariance Type:                   HC3                                         
                        coef    std err          z      P>|z|      [0.025      0

## 8. 주모형 결과표: wide format

In [27]:
KEY_TERMS = [
    AI,
    IT_ORG,
    DMI,
    INVEST_CONTROL,
    f"{AI}:{IT_ORG}",
    f"{AI}:{DMI}",
]
TERM_LABELS = {
    AI: "AI 활용 범위",
    IT_ORG: "정보화 담당 체계 보유",
    DMI: "디지털 성숙도(DMI)",
    INVEST_CONTROL: "정보화 투자 범위",
    f"{AI}:{IT_ORG}": "AI 활용 범위 × IT 조직",
    f"{AI}:{DMI}": "AI 활용 범위 × DMI",
}

wide_rows = []
for term in KEY_TERMS:
    row = {"변수": TERM_LABELS.get(term, term), "term": term}
    for model_key, spec in MODEL_SPECS.items():
        row[spec["column"]] = term_result(MODELS.get(model_key), term)["formatted"]
    wide_rows.append(row)

main_model_results_wide = pd.DataFrame(wide_rows)

fit_stat_rows = []
fit_stat_labels = [
    ("N", "N"),
    ("R²", "R²"),
    ("Adj. R²", "Adj. R²"),
    ("AIC", "AIC"),
    ("BIC", "BIC"),
    ("HC3 robust SE", "HC3 robust SE 적용 여부"),
    ("Controls", "Controls 포함 여부"),
    ("Industry FE", "Industry FE 포함 여부"),
    ("Region FE", "Region FE 포함 여부"),
    ("Firm type FE", "Firm type FE 포함 여부"),
    ("Investment control", "Investment control 포함 여부"),
]
fit_lookup = main_model_fit_stats.set_index("모형")
for label, col in fit_stat_labels:
    row = {"변수": label, "term": "fit_stat"}
    for model_key, spec in MODEL_SPECS.items():
        value = fit_lookup.loc[spec["short"], col]
        row[spec["column"]] = value
    fit_stat_rows.append(row)

main_model_results_wide_with_stats = pd.concat([main_model_results_wide, pd.DataFrame(fit_stat_rows)], ignore_index=True)

display(Markdown("[Table] 주모형 결과표: 핵심 변수 중심"))
display(main_model_results_wide_with_stats)
display(Markdown("주: 괄호 안은 HC3 robust standard error. * p < .05, ** p < .01, *** p < .001."))

[Table] 주모형 결과표: 핵심 변수 중심

,변수,term,Model 1: H1 Baseline,Model 2: H2 Basic: AI × IT org,Model 3: H2 Headline: Model D replication,Model 4: H3: AI × DMI,Model 5: Integrated complements
0,AI 활용 범위,ai_use_sum,0.052***\n(0.006),0.009\n(0.009),-0.024**\n(0.009),0.002\n(0.010),-0.029**\n(0.010)
1,정보화 담당 체계 보유,it_org_any,,0.116***\n(0.017),0.011\n(0.017),,0.011\n(0.018)
2,디지털 성숙도(DMI),dmi,,,,0.028***\n(0.002),0.015***\n(0.002)
3,정보화 투자 범위,it_invest_sum,,,0.108***\n(0.006),,0.094***\n(0.006)
4,AI 활용 범위 × IT 조직,ai_use_sum:it_org_any,,0.064***\n(0.011),0.057***\n(0.011),,0.060***\n(0.012)
5,AI 활용 범위 × DMI,ai_use_sum:dmi,,,,0.000\n(0.001),-0.001\n(0.001)
6,N,fit_stat,12203,12203,12203,12203,12203
7,R²,fit_stat,0.112,0.125,0.148,0.126,0.151
8,Adj. R²,fit_stat,0.11,0.122,0.145,0.124,0.148
9,AIC,fit_stat,26682.802,26517.309,26190.144,26493.169,26153.767


주: 괄호 안은 HC3 robust standard error. * p < .05, ** p < .01, *** p < .001.

## 9. 핵심 계수 요약표: long format

In [28]:
key_result_specs = [
    ("H1", "M1_H1_Baseline", AI, "AI 활용 범위와 DV의 기본 관련성"),
    ("H2 Basic", "M2_H2_Basic_AI_ITorg", f"{AI}:{IT_ORG}", "AI 활용 범위와 IT 조직 보유의 기본 결합 관련성"),
    ("H2 Headline", "M3_H2_Headline_Model_D", AI, "정보화 투자 범위 통제 후 AI 활용 범위의 단독 계수 방향"),
    ("H2 Headline", "M3_H2_Headline_Model_D", IT_ORG, "정보화 투자 범위 통제 후 IT 조직 보유의 단독 계수 방향"),
    ("H2 Headline", "M3_H2_Headline_Model_D", f"{AI}:{IT_ORG}", "정보화 투자 범위 통제 후 상호작용항 유지 여부 확인"),
    ("H3", "M4_H3_AI_DMI", f"{AI}:{DMI}", "AI 활용 범위와 DMI의 결합 관련성"),
    ("Integrated", "M5_Integrated_Complements", f"{AI}:{IT_ORG}", "두 보완재 동시 포함 후 AI × IT 조직 항 유지 여부"),
    ("Integrated", "M5_Integrated_Complements", f"{AI}:{DMI}", "두 보완재 동시 포함 후 AI × DMI 항 방향"),
]

key_rows = []
for hypothesis, model_key, term, note in key_result_specs:
    spec = MODEL_SPECS[model_key]
    key_rows.append(extract_key_results(model_key, MODELS.get(model_key), spec, term, hypothesis, note))

main_model_key_coefficients = pd.DataFrame(key_rows)
for col in ["계수", "HC3 Robust SE", "p-value", "R²", "Adj. R²"]:
    main_model_key_coefficients[col] = main_model_key_coefficients[col].round(4)

display(main_model_key_coefficients)

,가설,모형,종속변수,핵심 변수,계수,HC3 Robust SE,p-value,유의수준,N,R²,Adj. R²,해석 방향,비고
0,H1,H1 Baseline,effect_proc_improve,ai_use_sum,0.0521,0.0056,0.0000,***,12203,0.1125,0.1100,+,AI 활용 범위와 DV의 기본 관련성
1,H2 Basic,H2 Basic: AI × IT org,effect_proc_improve,ai_use_sum:it_org_any,0.0638,0.0109,0.0000,***,12203,0.1247,0.1221,+,AI 활용 범위와 IT 조직 보유의 기본 결합 관련성
2,H2 Headline,H2 Headline: Model D replication,effect_proc_improve,ai_use_sum,-0.0237,0.0090,0.0085,**,12203,0.1480,0.1454,-,정보화 투자 범위 통제 후 AI 활용 범위의 단독 계수 방향
3,H2 Headline,H2 Headline: Model D replication,effect_proc_improve,it_org_any,0.0110,0.0175,0.5274,,12203,0.1480,0.1454,+,정보화 투자 범위 통제 후 IT 조직 보유의 단독 계수 방향
4,H2 Headline,H2 Headline: Model D replication,effect_proc_improve,ai_use_sum:it_org_any,0.0575,0.0107,0.0000,***,12203,0.1480,0.1454,+,정보화 투자 범위 통제 후 상호작용항 유지 여부 확인
5,H3,H3: AI × DMI,effect_proc_improve,ai_use_sum:dmi,0.0004,0.0007,0.6233,,12203,0.1265,0.1239,+,AI 활용 범위와 DMI의 결합 관련성
6,Integrated,Integrated Complementarity Model,effect_proc_improve,ai_use_sum:it_org_any,0.0601,0.0119,0.0000,***,12203,0.1508,0.1481,+,두 보완재 동시 포함 후 AI × IT 조직 항 유지 여부
7,Integrated,Integrated Complementarity Model,effect_proc_improve,ai_use_sum:dmi,-0.0015,0.0008,0.0661,,12203,0.1508,0.1481,-,두 보완재 동시 포함 후 AI × DMI 항 방향


## 10. H2 Headline Model: 9주차 Model D 재현 결과

In [29]:
h2_headline_terms = [AI, IT_ORG, INVEST_CONTROL, f"{AI}:{IT_ORG}"]
H2_HEADLINE_NOTES = {
    AI: "정보화 투자 범위 통제 후 AI 활용 범위의 단독 관련성",
    IT_ORG: "정보화 투자 범위 통제 후 IT 조직 보유의 단독 관련성",
    INVEST_CONTROL: "정보화 투자 범위와 정보화 효과 인식의 관련성",
    f"{AI}:{IT_ORG}": "정보화 투자 범위 통제 후 AI 활용 범위와 IT 조직 보유의 결합 관련성",
}

h2_rows = []
h2_model = MODELS.get("M3_H2_Headline_Model_D")
for term in h2_headline_terms:
    tr = term_result(h2_model, term)
    h2_rows.append({
        "변수": term,
        "계수": tr["계수"],
        "HC3 Robust SE": tr["HC3 Robust SE"],
        "p-value": tr["p-value"],
        "유의수준": tr["유의수준"],
        "방향": tr["방향"],
        "해석 메모": H2_HEADLINE_NOTES[term],
    })

h2_headline_model_d_replication = pd.DataFrame(h2_rows)
for col in ["계수", "HC3 Robust SE", "p-value"]:
    h2_headline_model_d_replication[col] = h2_headline_model_d_replication[col].round(4)

display(Markdown("[Table] H2 Headline Model: 9주차 Model D 재현 결과"))
display(h2_headline_model_d_replication)

h2_interaction = term_result(h2_model, f"{AI}:{IT_ORG}")
display_note("H2 Headline 요약", [
    "Model 3은 9주차 Model D를 Pivot 이후 H2 headline model로 재배치한 결과이다.",
    f"정보화 투자 범위 통제 후 `{AI}:{IT_ORG}` 계수는 {h2_interaction['계수']:.3f}{h2_interaction['유의수준']}로 나타났다.",
    "따라서 H2에서는 단독 계수보다 AI 활용 범위와 IT 조직 보유의 결합 관련성이 유지되는지를 핵심적으로 확인한다.",
])

[Table] H2 Headline Model: 9주차 Model D 재현 결과

,변수,계수,HC3 Robust SE,p-value,유의수준,방향,해석 메모
0,ai_use_sum,-0.0237,0.0090,0.0085,**,-,정보화 투자 범위 통제 후 AI 활용 범위의 단독 관련성
1,it_org_any,0.0110,0.0175,0.5274,,+,정보화 투자 범위 통제 후 IT 조직 보유의 단독 관련성
2,it_invest_sum,0.1081,0.0058,0.0000,***,+,정보화 투자 범위와 정보화 효과 인식의 관련성
3,ai_use_sum:it_org_any,0.0575,0.0107,0.0000,***,+,정보화 투자 범위 통제 후 AI 활용 범위와 IT 조직 보유의 결합 관련성


**H2 Headline 요약**
- Model 3은 9주차 Model D를 Pivot 이후 H2 headline model로 재배치한 결과이다.
- 정보화 투자 범위 통제 후 `ai_use_sum:it_org_any` 계수는 0.057***로 나타났다.
- 따라서 H2에서는 단독 계수보다 AI 활용 범위와 IT 조직 보유의 결합 관련성이 유지되는지를 핵심적으로 확인한다.

## 11. 전체 계수표

In [30]:
full_coef_tables = []
for model_key, model in MODELS.items():
    spec = MODEL_SPECS[model_key]
    ci = model.conf_int()
    table = pd.DataFrame({
        "모형": spec["short"],
        "가설/역할": spec["label"],
        "term": model.params.index,
        "계수": model.params.values,
        "HC3 Robust SE": model.bse.values,
        "t": model.tvalues.values,
        "p-value": model.pvalues.values,
        "95% CI 하한": ci.iloc[:, 0].values,
        "95% CI 상한": ci.iloc[:, 1].values,
    })
    table["유의수준"] = table["p-value"].map(p_stars)
    full_coef_tables.append(table)

main_model_full_coefficients = pd.concat(full_coef_tables, ignore_index=True)
for col in ["계수", "HC3 Robust SE", "t", "p-value", "95% CI 하한", "95% CI 상한"]:
    main_model_full_coefficients[col] = main_model_full_coefficients[col].round(4)

display(main_model_full_coefficients)

,모형,가설/역할,term,계수,HC3 Robust SE,t,p-value,95% CI 하한,95% CI 상한,유의수준
0,Model 1,H1 Baseline,Intercept,3.7547,0.0429,87.5209,0.0000,3.6706,3.8388,***
1,Model 1,H1 Baseline,C(industry)[T.2],-0.0235,0.0370,-0.6367,0.5244,-0.0960,0.0489,
2,Model 1,H1 Baseline,C(industry)[T.3],-0.0420,0.0626,-0.6709,0.5023,-0.1646,0.0806,
3,Model 1,H1 Baseline,C(industry)[T.4],-0.0518,0.0400,-1.2959,0.1950,-0.1302,0.0266,
4,Model 1,H1 Baseline,C(industry)[T.5],0.0214,0.0412,0.5195,0.6034,-0.0594,0.1022,
...,...,...,...,...,...,...,...,...,...,...
182,Model 5,Integrated Complementarity Model,dmi,0.0151,0.0025,6.1597,0.0000,0.0103,0.0199,***
183,Model 5,Integrated Complementarity Model,it_invest_sum,0.0941,0.0063,14.8610,0.0000,0.0817,0.1065,***
184,Model 5,Integrated Complementarity Model,ai_use_sum:it_org_any,0.0601,0.0119,5.0672,0.0000,0.0368,0.0833,***
185,Model 5,Integrated Complementarity Model,ai_use_sum:dmi,-0.0015,0.0008,-1.8378,0.0661,-0.0030,0.0001,


## 12. 보고서용 해석 메모 초안

In [31]:
def term_sentence(model_key: str, term: str, label: str) -> str:
    tr = term_result(MODELS.get(model_key), term)
    if pd.isna(tr["계수"]):
        return f"{label}은 해당 모형에서 추정되지 않았다."
    sig_text = "유의한 " if tr["유의수준"] else "통계적으로 뚜렷하지 않은 "
    direction_text = "양(+)" if tr["방향"] == "+" else "음(-)" if tr["방향"] == "-" else "0"
    return f"{label}은 {sig_text}{direction_text}의 계수({tr['계수']:.3f}{tr['유의수준']}, p={tr['p-value']:.4f})로 나타났다."

interpretation_lines = [
    term_sentence("M1_H1_Baseline", AI, "H1에서 AI 활용 범위"),
    term_sentence("M2_H2_Basic_AI_ITorg", f"{AI}:{IT_ORG}", "H2 Basic에서 AI 활용 범위 × IT 조직 보유 상호작용항"),
    "Model 3은 9주차 Model D를 Pivot 이후 H2 headline model로 재배치한 결과이다.",
    term_sentence("M3_H2_Headline_Model_D", AI, "H2 Headline에서 AI 활용 범위 단독항"),
    term_sentence("M3_H2_Headline_Model_D", IT_ORG, "H2 Headline에서 IT 조직 보유 단독항"),
    term_sentence("M3_H2_Headline_Model_D", f"{AI}:{IT_ORG}", "H2 Headline에서 AI 활용 범위 × IT 조직 보유 상호작용항"),
    term_sentence("M4_H3_AI_DMI", f"{AI}:{DMI}", "H3에서 AI 활용 범위 × DMI 상호작용항"),
    term_sentence("M5_Integrated_Complements", f"{AI}:{IT_ORG}", "통합 모형에서 AI 활용 범위 × IT 조직 보유 상호작용항"),
    term_sentence("M5_Integrated_Complements", f"{AI}:{DMI}", "통합 모형에서 AI 활용 범위 × DMI 상호작용항"),
    "이 결과는 인과효과가 아니라 통제변수를 포함한 OLS HC3 모형에서 관찰되는 조건부 연관성으로 해석한다.",
]

main_model_interpretation_notes = "\n".join(f"- {line}" for line in interpretation_lines)
display(Markdown(main_model_interpretation_notes))

- H1에서 AI 활용 범위은 유의한 양(+)의 계수(0.052***, p=0.0000)로 나타났다.
- H2 Basic에서 AI 활용 범위 × IT 조직 보유 상호작용항은 유의한 양(+)의 계수(0.064***, p=0.0000)로 나타났다.
- Model 3은 9주차 Model D를 Pivot 이후 H2 headline model로 재배치한 결과이다.
- H2 Headline에서 AI 활용 범위 단독항은 유의한 음(-)의 계수(-0.024**, p=0.0085)로 나타났다.
- H2 Headline에서 IT 조직 보유 단독항은 통계적으로 뚜렷하지 않은 양(+)의 계수(0.011, p=0.5274)로 나타났다.
- H2 Headline에서 AI 활용 범위 × IT 조직 보유 상호작용항은 유의한 양(+)의 계수(0.057***, p=0.0000)로 나타났다.
- H3에서 AI 활용 범위 × DMI 상호작용항은 통계적으로 뚜렷하지 않은 양(+)의 계수(0.000, p=0.6233)로 나타났다.
- 통합 모형에서 AI 활용 범위 × IT 조직 보유 상호작용항은 유의한 양(+)의 계수(0.060***, p=0.0000)로 나타났다.
- 통합 모형에서 AI 활용 범위 × DMI 상호작용항은 통계적으로 뚜렷하지 않은 음(-)의 계수(-0.001, p=0.0661)로 나타났다.
- 이 결과는 인과효과가 아니라 통제변수를 포함한 OLS HC3 모형에서 관찰되는 조건부 연관성으로 해석한다.

## 13. 저장

In [32]:
if SAVE_OUTPUTS:
    TABLE_DIR.mkdir(parents=True, exist_ok=True)
    MODEL_DIR.mkdir(parents=True, exist_ok=True)

    main_model_results_wide_with_stats.to_csv(TABLE_DIR / "main_model_results_wide.csv", index=False, encoding="utf-8-sig")
    main_model_results_wide_with_stats.to_excel(TABLE_DIR / "main_model_results_wide.xlsx", index=False)

    main_model_key_coefficients.to_csv(TABLE_DIR / "main_model_key_coefficients.csv", index=False, encoding="utf-8-sig")
    main_model_key_coefficients.to_excel(TABLE_DIR / "main_model_key_coefficients.xlsx", index=False)

    main_model_fit_stats.to_csv(TABLE_DIR / "main_model_fit_stats.csv", index=False, encoding="utf-8-sig")
    main_model_fit_stats.to_excel(TABLE_DIR / "main_model_fit_stats.xlsx", index=False)

    h2_headline_model_d_replication.to_csv(TABLE_DIR / "h2_headline_model_d_replication.csv", index=False, encoding="utf-8-sig")
    h2_headline_model_d_replication.to_excel(TABLE_DIR / "h2_headline_model_d_replication.xlsx", index=False)

    full_summary_text = []
    for model_key, model in MODELS.items():
        spec = MODEL_SPECS[model_key]
        full_summary_text.append("=" * 100)
        full_summary_text.append(f"{spec['short']} {spec['label']}")
        full_summary_text.append(spec["formula"])
        full_summary_text.append(str(model.summary()))
    (MODEL_DIR / "main_model_full_summary.txt").write_text("\n\n".join(full_summary_text), encoding="utf-8")

    (OUTPUT_DIR / "main_model_interpretation_notes.txt").write_text(main_model_interpretation_notes, encoding="utf-8")

    with pd.ExcelWriter(OUTPUT_DIR / "main_model_summary.xlsx") as writer:
        main_model_results_wide_with_stats.to_excel(writer, sheet_name="wide_results", index=False)
        main_model_key_coefficients.to_excel(writer, sheet_name="key_coefficients", index=False)
        main_model_fit_stats.to_excel(writer, sheet_name="fit_stats", index=False)
        h2_headline_model_d_replication.to_excel(writer, sheet_name="h2_headline", index=False)
        main_model_full_coefficients.to_excel(writer, sheet_name="full_coefficients", index=False)
        model_spec_table.to_excel(writer, sheet_name="model_specs", index=False)

    print("저장 완료:", OUTPUT_DIR.relative_to(BASE_DIR))
else:
    print("SAVE_OUTPUTS=False이므로 파일 저장은 수행하지 않았습니다.")

SAVE_OUTPUTS=False이므로 파일 저장은 수행하지 않았습니다.


## 14. 실행 요약

In [33]:
print("사용 데이터 파일:", DATA_PATH.relative_to(BASE_DIR))
print("원 데이터 shape:", LOADED_SHAPE)
print("추정 모형 수:", len(MODELS))
print("모형별 N:")
for model_key, model in MODELS.items():
    print(f"- {MODEL_SPECS[model_key]['short']} {MODEL_SPECS[model_key]['label']}: N={int(model.nobs):,}")
print("공분산 추정:", "HC3 robust standard errors")
print("H2 headline model:", "Model 3: H2 Headline: Model D replication")
print("SAVE_OUTPUTS:", SAVE_OUTPUTS)
if SAVE_OUTPUTS:
    print("결과 저장 폴더:", OUTPUT_DIR.relative_to(BASE_DIR))

사용 데이터 파일: working/analysis/nia_2024_analysis_total.csv
원 데이터 shape: (12203, 103)
추정 모형 수: 5
모형별 N:
- Model 1 H1 Baseline: N=12,203
- Model 2 H2 Basic: AI × IT org: N=12,203
- Model 3 H2 Headline: Model D replication: N=12,203
- Model 4 H3: AI × DMI: N=12,203
- Model 5 Integrated Complementarity Model: N=12,203
공분산 추정: HC3 robust standard errors
H2 headline model: Model 3: H2 Headline: Model D replication
SAVE_OUTPUTS: False


## Robustness Check: Alternative Outcome Dimensions for H2 Headline

기존 H2 headline Model 3 specification을 유지하되, 종속변수만 `effect_innov_outcome`, `effect_decision_improve`로 바꾸어 AI 활용 범위 단독항과 `AI 활용 범위 × IT 조직 보유` 상호작용항의 방향 및 유의성을 점검한다.

In [34]:
ALTERNATIVE_DVS = ["effect_innov_outcome", "effect_decision_improve"]
MODEL3_ALT_REQUIRED_VARS = ALTERNATIVE_DVS + [
    AI,
    IT_ORG,
    INVEST_CONTROL,
    "firm_size",
    "industry",
    "region",
    "firm_type",
]

model3_alt_variable_check = pd.DataFrame([
    {
        "variable": var,
        "exists": var in df.columns,
        "valid_N": int(df[var].notna().sum()) if var in df.columns else np.nan,
        "missing_N": int(df[var].isna().sum()) if var in df.columns else np.nan,
        "dtype": str(df[var].dtype) if var in df.columns else "missing",
    }
    for var in MODEL3_ALT_REQUIRED_VARS
])

display(model3_alt_variable_check)

missing_model3_alt_vars = model3_alt_variable_check.loc[
    ~model3_alt_variable_check["exists"], "variable"
].tolist()

if missing_model3_alt_vars:
    missing_text = ", ".join(missing_model3_alt_vars)
    display(Markdown(f"**Alternative DV robustness check 중단:** 누락 변수: `{missing_text}`"))
    raise ValueError("Missing variables for Model 3 alternative DV robustness check: " + missing_text)

MODEL3_ALT_CONTROLS = ["firm_size", "industry", "region", "firm_type"]
MODEL3_ALT_CONTROL_TERMS = control_formula(MODEL3_ALT_CONTROLS)
MODEL3_ALT_CONTROL_SUFFIX = f" + {MODEL3_ALT_CONTROL_TERMS}" if MODEL3_ALT_CONTROL_TERMS else ""
MODEL3_ALT_INTERACTION = f"{AI}:{IT_ORG}"

print("Alternative DV robustness check validation passed.")
print("Controls:", MODEL3_ALT_CONTROLS)
print("Interaction term:", MODEL3_ALT_INTERACTION)


,variable,exists,valid_N,missing_N,dtype
0,effect_innov_outcome,True,12203,0,int64
1,effect_decision_improve,True,12203,0,int64
2,ai_use_sum,True,12203,0,int64
3,it_org_any,True,12203,0,int64
4,it_invest_sum,True,12203,0,int64
5,firm_size,True,12203,0,int64
6,industry,True,12203,0,int64
7,region,True,12203,0,int64
8,firm_type,True,12203,0,int64


Alternative DV robustness check validation passed.
Controls: ['firm_size', 'industry', 'region', 'firm_type']
Interaction term: ai_use_sum:it_org_any


In [35]:
def p_stars_with_dagger(p_value: float) -> str:
    """Significance stars for the alternative DV robustness check."""
    if pd.isna(p_value):
        return ""
    if p_value < 0.001:
        return "***"
    if p_value < 0.01:
        return "**"
    if p_value < 0.05:
        return "*"
    if p_value < 0.1:
        return "†"
    return ""


def sig_label(p_value: float) -> str:
    stars = p_stars_with_dagger(p_value)
    return stars if stars else "n.s."


def direction_label(coef: float) -> str:
    if pd.isna(coef):
        return ""
    if coef > 0:
        return "+"
    if coef < 0:
        return "-"
    return "0"


def headline_pattern_status(ai_coef: float, interaction_coef: float, interaction_pvalue: float) -> str:
    conditions = [
        ai_coef < 0,
        interaction_coef > 0,
        interaction_pvalue < 0.05,
    ]
    if all(conditions):
        return "Maintained"
    if any(conditions):
        return "Partially maintained"
    return "Not maintained"


def run_model3_for_dv(dv_name: str):
    """Run the original Model 3 specification with an alternative dependent variable."""
    model_vars = [dv_name, AI, IT_ORG, INVEST_CONTROL] + MODEL3_ALT_CONTROLS
    missing_vars = [var for var in model_vars if var not in df.columns]
    if missing_vars:
        raise ValueError(f"{dv_name} 모형 추정 불가. 누락 변수: {', '.join(missing_vars)}")

    model_data = df[model_vars].copy()
    for var in model_vars:
        model_data[var] = pd.to_numeric(model_data[var], errors="coerce")
    model_data = model_data.dropna(subset=model_vars).copy()

    if model_data.empty:
        raise ValueError(f"{dv_name} 모형 추정 불가. complete-case 데이터가 없습니다.")

    formula = (
        f"{dv_name} ~ {AI} + {IT_ORG} + {INVEST_CONTROL} + "
        f"{MODEL3_ALT_INTERACTION}{MODEL3_ALT_CONTROL_SUFFIX}"
    )
    result = smf.ols(formula, data=model_data).fit(cov_type="HC3")
    return result, model_data, formula


def extract_model3_alt_summary(dv_name: str, result) -> dict:
    ai_coef = float(result.params[AI])
    ai_se = float(result.bse[AI])
    ai_pvalue = float(result.pvalues[AI])
    interaction_coef = float(result.params[MODEL3_ALT_INTERACTION])
    interaction_se = float(result.bse[MODEL3_ALT_INTERACTION])
    interaction_pvalue = float(result.pvalues[MODEL3_ALT_INTERACTION])

    return {
        "dv_name": dv_name,
        "model_name": "Model 3 Alternative DV",
        "N": int(result.nobs),
        "R_squared": float(result.rsquared),
        "Adj_R_squared": float(result.rsquared_adj),
        "ai_use_sum_coef": ai_coef,
        "ai_use_sum_se": ai_se,
        "ai_use_sum_pvalue": ai_pvalue,
        "ai_use_sum_sig": p_stars_with_dagger(ai_pvalue),
        "ai_use_sum_direction": direction_label(ai_coef),
        "interaction_coef": interaction_coef,
        "interaction_se": interaction_se,
        "interaction_pvalue": interaction_pvalue,
        "interaction_sig": p_stars_with_dagger(interaction_pvalue),
        "interaction_direction": direction_label(interaction_coef),
        "headline_pattern 유지 여부": headline_pattern_status(
            ai_coef, interaction_coef, interaction_pvalue
        ),
    }


def build_model3_alt_summary_table(dv_names: list[str]) -> tuple[pd.DataFrame, dict, dict]:
    rows = []
    results = {}
    formulas = {}
    for dv_name in dv_names:
        result, model_data, formula = run_model3_for_dv(dv_name)
        results[dv_name] = result
        formulas[dv_name] = formula
        rows.append(extract_model3_alt_summary(dv_name, result))
        print(
            f"{dv_name} Model 3 alternative DV 추정 완료: "
            f"N={int(result.nobs):,}, R²={result.rsquared:.3f}, "
            f"Adj. R²={result.rsquared_adj:.3f}"
        )
    return pd.DataFrame(rows), results, formulas


def build_model3_alt_display_table(summary_df: pd.DataFrame) -> pd.DataFrame:
    display_df = summary_df.copy()
    display_df["DV"] = display_df["dv_name"]
    display_df["ai_use_sum 계수"] = display_df.apply(
        lambda row: f"{row['ai_use_sum_coef']:.3f}{row['ai_use_sum_sig']}", axis=1
    )
    display_df["ai_use_sum 유의성"] = display_df["ai_use_sum_pvalue"].map(sig_label)
    display_df["ai_use_sum 방향"] = display_df["ai_use_sum_direction"]
    display_df["ai_use_sum × it_org_any 계수"] = display_df.apply(
        lambda row: f"{row['interaction_coef']:.3f}{row['interaction_sig']}", axis=1
    )
    display_df["상호작용항 유의성"] = display_df["interaction_pvalue"].map(sig_label)
    display_df["상호작용항 방향"] = display_df["interaction_direction"]
    display_df["H2 headline 유지 여부"] = display_df["headline_pattern 유지 여부"]
    return display_df[[
        "DV",
        "ai_use_sum 계수",
        "ai_use_sum 유의성",
        "ai_use_sum 방향",
        "ai_use_sum × it_org_any 계수",
        "상호작용항 유의성",
        "상호작용항 방향",
        "H2 headline 유지 여부",
    ]]


def korean_term_description(row: pd.Series, prefix: str) -> str:
    coef = row[f"{prefix}_coef"]
    pvalue = row[f"{prefix}_pvalue"]
    stars = row[f"{prefix}_sig"]
    direction_text = "양(+)" if coef > 0 else "음(-)" if coef < 0 else "0"
    if pvalue < 0.05:
        sig_text = "통계적으로 유의한"
    elif pvalue < 0.1:
        sig_text = "10% 수준에서 약하게 유의한"
    else:
        sig_text = "통계적으로 유의하지 않은"
    return f"{direction_text}의 {sig_text} 계수({coef:.3f}{stars}, p={pvalue:.4f})"


def build_model3_alt_interpretation(summary_df: pd.DataFrame) -> str:
    rows = {row["dv_name"]: row for _, row in summary_df.iterrows()}
    innov = rows["effect_innov_outcome"]
    decision = rows["effect_decision_improve"]
    both_maintained = all(
        row["headline_pattern 유지 여부"] == "Maintained"
        for _, row in summary_df.iterrows()
    )

    if both_maintained:
        overall = "두 대체 종속변수 모두에서 H2 headline pattern이 유지된다."
    else:
        statuses = ", ".join(
            f"{row['dv_name']}={row['headline_pattern 유지 여부']}"
            for _, row in summary_df.iterrows()
        )
        overall = (
            "두 대체 종속변수 모두에서 H2 headline pattern이 완전히 유지되지는 않는다 "
            f"({statuses}). 따라서 프로세스 개선 효과 인식에서는 강하지만 "
            "다른 성과 차원에서는 제한적으로 나타난다고 정직하게 해석할 수 있다."
        )

    return "\n".join([
        "**자동 해석**",
        f"- `effect_innov_outcome`에서 `ai_use_sum` 단독항은 {korean_term_description(innov, 'ai_use_sum')}로 나타났다.",
        f"- `effect_innov_outcome`에서 `ai_use_sum × it_org_any` 상호작용항은 {korean_term_description(innov, 'interaction')}로 나타났다.",
        f"- `effect_decision_improve`에서 `ai_use_sum` 단독항은 {korean_term_description(decision, 'ai_use_sum')}로 나타났다.",
        f"- `effect_decision_improve`에서 `ai_use_sum × it_org_any` 상호작용항은 {korean_term_description(decision, 'interaction')}로 나타났다.",
        f"- {overall}",
    ])


model3_alternative_dv_summary, MODEL3_ALT_RESULTS, MODEL3_ALT_FORMULAS = build_model3_alt_summary_table(ALTERNATIVE_DVS)
model3_alternative_dv_display = build_model3_alt_display_table(model3_alternative_dv_summary)
model3_alternative_dv_interpretation = build_model3_alt_interpretation(model3_alternative_dv_summary)

numeric_cols = [
    "R_squared",
    "Adj_R_squared",
    "ai_use_sum_coef",
    "ai_use_sum_se",
    "ai_use_sum_pvalue",
    "interaction_coef",
    "interaction_se",
    "interaction_pvalue",
]
model3_alternative_dv_summary_display = model3_alternative_dv_summary.copy()
for col in numeric_cols:
    model3_alternative_dv_summary_display[col] = model3_alternative_dv_summary_display[col].round(4)

display(model3_alternative_dv_summary_display)
display(model3_alternative_dv_display)
display(Markdown("주: 괄호 밖 표기는 계수와 유의수준. † p < .1, * p < .05, ** p < .01, *** p < .001. 표준오차는 상세표의 HC3 robust SE를 참조."))
display(Markdown(model3_alternative_dv_interpretation))


effect_innov_outcome Model 3 alternative DV 추정 완료: N=12,203, R²=0.222, Adj. R²=0.219
effect_decision_improve Model 3 alternative DV 추정 완료: N=12,203, R²=0.177, Adj. R²=0.174


,dv_name,model_name,N,R_squared,Adj_R_squared,ai_use_sum_coef,ai_use_sum_se,ai_use_sum_pvalue,ai_use_sum_sig,ai_use_sum_direction,interaction_coef,interaction_se,interaction_pvalue,interaction_sig,interaction_direction,headline_pattern 유지 여부
0,effect_innov_outcome,Model 3 Alternative DV,12203,0.2218,0.2194,-0.0382,0.0092,0.0,***,-,0.0974,0.0112,0.0,***,+,Maintained
1,effect_decision_improve,Model 3 Alternative DV,12203,0.1767,0.1742,-0.0350,0.0081,0.0,***,-,0.0849,0.0104,0.0,***,+,Maintained


,DV,ai_use_sum 계수,ai_use_sum 유의성,ai_use_sum 방향,ai_use_sum × it_org_any 계수,상호작용항 유의성,상호작용항 방향,H2 headline 유지 여부
0,effect_innov_outcome,-0.038***,***,-,0.097***,***,+,Maintained
1,effect_decision_improve,-0.035***,***,-,0.085***,***,+,Maintained


주: 괄호 밖 표기는 계수와 유의수준. † p < .1, * p < .05, ** p < .01, *** p < .001. 표준오차는 상세표의 HC3 robust SE를 참조.

**자동 해석**
- `effect_innov_outcome`에서 `ai_use_sum` 단독항은 음(-)의 통계적으로 유의한 계수(-0.038***, p=0.0000)로 나타났다.
- `effect_innov_outcome`에서 `ai_use_sum × it_org_any` 상호작용항은 양(+)의 통계적으로 유의한 계수(0.097***, p=0.0000)로 나타났다.
- `effect_decision_improve`에서 `ai_use_sum` 단독항은 음(-)의 통계적으로 유의한 계수(-0.035***, p=0.0000)로 나타났다.
- `effect_decision_improve`에서 `ai_use_sum × it_org_any` 상호작용항은 양(+)의 통계적으로 유의한 계수(0.085***, p=0.0000)로 나타났다.
- 두 대체 종속변수 모두에서 H2 headline pattern이 유지된다.

In [36]:
if SAVE_OUTPUTS:
    alt_output_dir = BASE_DIR / "outputs"
    alt_output_dir.mkdir(parents=True, exist_ok=True)
    alt_csv_path = alt_output_dir / "08_main_model/model3_alternative_dv_summary.csv"
    alt_xlsx_path = alt_output_dir / "08_main_model/model3_alternative_dv_summary.xlsx"

    model3_alternative_dv_summary.to_csv(alt_csv_path, index=False, encoding="utf-8-sig")
    with pd.ExcelWriter(alt_xlsx_path) as writer:
        model3_alternative_dv_summary.to_excel(writer, sheet_name="detailed_summary", index=False)
        model3_alternative_dv_display.to_excel(writer, sheet_name="paper_table", index=False)

    print("Alternative DV summary 저장 완료:")
    print("-", alt_csv_path.relative_to(BASE_DIR))
    print("-", alt_xlsx_path.relative_to(BASE_DIR))
else:
    print("SAVE_OUTPUTS=False이므로 alternative DV summary 파일 저장은 수행하지 않았습니다.")


SAVE_OUTPUTS=False이므로 alternative DV summary 파일 저장은 수행하지 않았습니다.


## Appendix Table 4-1: Main Model Key Results

주모형 결과를 부록용 compact coefficient table로 정리한다. Dummy variable 계수는 표에서 생략하고, 포함 여부만 하단 행에 표시한다.

In [37]:
APPENDIX_A1_TITLE = "[Appendix Table A1] 주모형 핵심 회귀결과"
APPENDIX_A1_NOTE = (
    "주: 괄호 안은 HC3 robust standard errors. 모든 모형은 기업 규모, 업종, 지역, 기업유형을 통제하였다. "
    "업종·지역·기업유형 더미 계수는 지면상 생략하였다. † p<.10, * p<.05, ** p<.01, *** p<.001."
)

APPENDIX_A1_MODEL_ORDER = [
    ("M1_H1_Baseline", "Model 1 H1 Baseline"),
    ("M2_H2_Basic_AI_ITorg", "Model 2 H2 Basic: AI × IT org"),
    ("M3_H2_Headline_Model_D", "Model 3 H2 Headline"),
    ("M4_H3_AI_DMI", "Model 4 H3: AI × DMI"),
    ("M5_Integrated_Complements", "Model 5 Integrated"),
]

APPENDIX_A1_TERMS = [
    ("AI 활용 범위", AI),
    ("정보화 담당 체계 보유", IT_ORG),
    ("디지털 성숙도", DMI),
    ("정보화 투자 범위", INVEST_CONTROL),
    ("AI 활용 범위 × IT 조직", f"{AI}:{IT_ORG}"),
    ("AI 활용 범위 × DMI", f"{AI}:{DMI}"),
]


def appendix_a1_stars(p_value: float) -> str:
    if pd.isna(p_value):
        return ""
    if p_value < 0.001:
        return "***"
    if p_value < 0.01:
        return "**"
    if p_value < 0.05:
        return "*"
    if p_value < 0.10:
        return "†"
    return ""


def appendix_a1_coef_cell(model, term: str) -> str:
    if model is None or term not in model.params.index:
        return ""
    coef = float(model.params[term])
    se = float(model.bse[term])
    p_value = float(model.pvalues[term])
    return f"{coef:.3f}{appendix_a1_stars(p_value)} ({se:.3f})"


def appendix_a1_yes_no(condition: bool) -> str:
    return "Yes" if condition else "No"


def build_appendix_a1_table() -> pd.DataFrame:
    rows = []

    for label, term in APPENDIX_A1_TERMS:
        row = {"변수": label}
        for model_key, column_name in APPENDIX_A1_MODEL_ORDER:
            row[column_name] = appendix_a1_coef_cell(MODELS.get(model_key), term)
        rows.append(row)

    fit_rows = [
        ("N", lambda model_key, model, spec: f"{int(model.nobs):,}" if model is not None else ""),
        ("R²", lambda model_key, model, spec: f"{model.rsquared:.3f}" if model is not None else ""),
        ("Adj. R²", lambda model_key, model, spec: f"{model.rsquared_adj:.3f}" if model is not None else ""),
        ("Controls", lambda model_key, model, spec: appendix_a1_yes_no(bool(available_controls))),
        ("Industry FE", lambda model_key, model, spec: appendix_a1_yes_no("industry" in available_controls)),
        ("Region FE", lambda model_key, model, spec: appendix_a1_yes_no("region" in available_controls)),
        ("Firm type FE", lambda model_key, model, spec: appendix_a1_yes_no("firm_type" in available_controls)),
        ("Investment control", lambda model_key, model, spec: appendix_a1_yes_no(spec.get("investment_control") == "Yes")),
    ]

    for label, getter in fit_rows:
        row = {"변수": label}
        for model_key, column_name in APPENDIX_A1_MODEL_ORDER:
            row[column_name] = getter(model_key, MODELS.get(model_key), MODEL_SPECS[model_key])
        rows.append(row)

    return pd.DataFrame(rows, columns=["변수"] + [name for _, name in APPENDIX_A1_MODEL_ORDER])


appendix_table_a1_main_model_key_results = build_appendix_a1_table()

display(Markdown(f"**{APPENDIX_A1_TITLE}**"))
display(appendix_table_a1_main_model_key_results)
display(Markdown(APPENDIX_A1_NOTE))

# 검증: Model 3 핵심 값 확인
model3 = MODELS["M3_H2_Headline_Model_D"]
assert np.isclose(float(model3.params[AI]), -0.024, atol=0.002), f"Model 3 {AI} 값 확인 필요: {model3.params[AI]}"
assert np.isclose(float(model3.params[f"{AI}:{IT_ORG}"]), 0.0575, atol=0.002), f"Model 3 {AI}:{IT_ORG} 값 확인 필요: {model3.params[f'{AI}:{IT_ORG}']}"
assert int(model3.nobs) == 12203, f"Model 3 N 확인 필요: {model3.nobs}"
assert np.isclose(float(model3.rsquared), 0.148, atol=0.002), f"Model 3 R² 확인 필요: {model3.rsquared}"

print("Appendix A1 validation passed:")
print(f"- Model 3 {AI}: {model3.params[AI]:.4f}")
print(f"- Model 3 {AI}:{IT_ORG}: {model3.params[f'{AI}:{IT_ORG}']:.4f}")
print(f"- Model 3 N: {int(model3.nobs):,}")
print(f"- Model 3 R²: {model3.rsquared:.4f}")


**[Appendix Table A1] 주모형 핵심 회귀결과**

,변수,Model 1 H1 Baseline,Model 2 H2 Basic: AI × IT org,Model 3 H2 Headline,Model 4 H3: AI × DMI,Model 5 Integrated
0,AI 활용 범위,0.052*** (0.006),0.009 (0.009),-0.024** (0.009),0.002 (0.010),-0.029** (0.010)
1,정보화 담당 체계 보유,,0.116*** (0.017),0.011 (0.017),,0.011 (0.018)
2,디지털 성숙도,,,,0.028*** (0.002),0.015*** (0.002)
3,정보화 투자 범위,,,0.108*** (0.006),,0.094*** (0.006)
4,AI 활용 범위 × IT 조직,,0.064*** (0.011),0.057*** (0.011),,0.060*** (0.012)
5,AI 활용 범위 × DMI,,,,0.000 (0.001),-0.001† (0.001)
6,N,"12,203","12,203","12,203","12,203","12,203"
7,R²,0.112,0.125,0.148,0.126,0.151
8,Adj. R²,0.110,0.122,0.145,0.124,0.148
9,Controls,Yes,Yes,Yes,Yes,Yes


주: 괄호 안은 HC3 robust standard errors. 모든 모형은 기업 규모, 업종, 지역, 기업유형을 통제하였다. 업종·지역·기업유형 더미 계수는 지면상 생략하였다. † p<.10, * p<.05, ** p<.01, *** p<.001.

Appendix A1 validation passed:
- Model 3 ai_use_sum: -0.0237
- Model 3 ai_use_sum:it_org_any: 0.0575
- Model 3 N: 12,203
- Model 3 R²: 0.1480


In [38]:
if SAVE_OUTPUTS:
    TABLE_DIR.mkdir(parents=True, exist_ok=True)
    appendix_a1_csv_path = TABLE_DIR / "appendix_table_a1_main_model_key_results.csv"
    appendix_a1_xlsx_path = TABLE_DIR / "appendix_table_a1_main_model_key_results.xlsx"

    appendix_table_a1_main_model_key_results.to_csv(appendix_a1_csv_path, index=False, encoding="utf-8-sig")

    with pd.ExcelWriter(appendix_a1_xlsx_path, engine="openpyxl") as writer:
        appendix_table_a1_main_model_key_results.to_excel(
            writer,
            sheet_name="Appendix Table A1",
            index=False,
            startrow=2,
        )
        ws = writer.sheets["Appendix Table A1"]
        ws["A1"] = APPENDIX_A1_TITLE
        ws["A1"].font = ws["A1"].font.copy(bold=True, size=13)

        header_row = 3
        note_row = header_row + len(appendix_table_a1_main_model_key_results) + 3
        ws.cell(row=note_row, column=1, value=APPENDIX_A1_NOTE)

        from openpyxl.styles import Alignment, Font, PatternFill

        model3_fill = PatternFill(fill_type="solid", fgColor="FFF2CC")
        header_fill = PatternFill(fill_type="solid", fgColor="D9EAF7")

        for cell in ws[header_row]:
            cell.font = Font(bold=True)
            cell.fill = header_fill
            cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

        model3_col_idx = list(appendix_table_a1_main_model_key_results.columns).index("Model 3 H2 Headline") + 1
        for row in range(1, note_row + 1):
            ws.cell(row=row, column=model3_col_idx).fill = model3_fill

        ws.column_dimensions["A"].width = 28
        for col_idx in range(2, len(appendix_table_a1_main_model_key_results.columns) + 1):
            col_letter = ws.cell(row=header_row, column=col_idx).column_letter
            ws.column_dimensions[col_letter].width = 24
            for row_idx in range(header_row, note_row + 1):
                ws.cell(row=row_idx, column=col_idx).alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

        ws.cell(row=note_row, column=1).alignment = Alignment(wrap_text=True, vertical="top")
        ws.cell(row=note_row, column=1).font = Font(italic=True)
        ws.merge_cells(start_row=note_row, start_column=1, end_row=note_row, end_column=len(appendix_table_a1_main_model_key_results.columns))

        note_df = pd.DataFrame({"note": [APPENDIX_A1_NOTE]})
        note_df.to_excel(writer, sheet_name="Notes", index=False)

    print("Appendix A1 저장 완료:")
    print("-", appendix_a1_csv_path.relative_to(BASE_DIR))
    print("-", appendix_a1_xlsx_path.relative_to(BASE_DIR))
else:
    print("SAVE_OUTPUTS=False이므로 Appendix A1 파일 저장은 수행하지 않았습니다.")


SAVE_OUTPUTS=False이므로 Appendix A1 파일 저장은 수행하지 않았습니다.
